Import Libraries 
    The cell imports all neccssary libaries that i will be using for data manipulation, Cleaning, graph construction and Machine learning

In [122]:

# 1


import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import networkx as nx
import ast


Loading and Preparing the physical Dataset 
    1: load each csv file
        -use pandas.read_csv() and parse the bucket column as a datetime since it represents the timestamp of each measurement.
         This ensures proper time alignment later when combining with network data.
    2: add a agg_window colum
        -Each dataset is tagged with the aggregation window it came from (10s, 16s, or 30s).
         This provides useful metadata and helps distinguish rows originating from different sampling frequencies. (this is what the paper does)
    3: concatinate all physical data into a single dataframe.
    4: inspect all colums and check for duplicates
        -print the set of columns present in each physical file (to confirm schema consistency)
        -print the number of duplicate timestamps in the combined physical dataset
        -Duplicate timestamps are expected because many sensors report measurements at the same moment.
         These will be resolved later

In [123]:

# 2

phy10s = pd.read_csv('../../exports/data/phys_agg_30s.csv',
    parse_dates=["bucket"],
)

phy10s["agg_window"] = 10


phyDf = pd.concat([phy10s], ignore_index=True)

print("Physical columns per file:")
print(set(phy10s.columns))

print("Physical duplicate buckets:", phyDf["bucket"].duplicated().sum())
display(phyDf.head())


Physical columns per file:
{'bucket', 'max_value', 'num_measurements', 'agg_window', 'stddev_value', 'system_id', 'asset_id', 'prop_key', 'avg_value', 'attack_types', 'num_attacks', 'min_value'}
Physical duplicate buckets: 14391


,bucket,system_id,prop_key,asset_id,avg_value,max_value,min_value,stddev_value,num_measurements,num_attacks,attack_types,agg_window
0,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_1,31.8,178.0,0.0,61.926121,10,0,['normal'],10
1,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_2,0.0,0.0,0.0,0.000000,10,0,['normal'],10
2,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_3,0.0,0.0,0.0,0.000000,10,0,['normal'],10
3,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_4,0.0,0.0,0.0,0.000000,10,0,['normal'],10
4,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_5,0.0,0.0,0.0,0.000000,10,0,['normal'],10


Loading and preparing network datasets 
    1: Load each newtork csv
        -We read each file using pandas.read_csv() and parse the bucket column as a datetime value.
         This ensures proper time-based alignment when combining physical and network layers.
    2: add a agg_window colum 
        -Each dataset is tagged with its aggregation interval (10s, 16s, or 30s).
         This allows the model to learn patterns that may vary by sampling frequency.
    3: concatinate all 3 datasets into one
    4: inspect colums for duplicates 
        -Counting duplicate timestamps reveals how many rows share the same time bucket, which is expected in network data where several devices may     communicate simultaneously.
     

In [124]:

# 3


network30s = pd.read_csv("../../exports/data/scada_resolved_agg_30s.csv",
    parse_dates=["bucket"],
)

network30s["agg_window"] = 30

networkDf = pd.concat([network30s], ignore_index=True)

print("\nNetwork columns per file:")
print(set(network30s.columns))

print("Network duplicate buckets:", networkDf["bucket"].duplicated().sum())
display(networkDf.head())



Network columns per file:
{'bucket', 'tcp_fin_count', 'system_id', 'tcp_syn_count', 'destination_total_packets', 'destination_key', 'tcp_syn_ratio', 'destination_mac', 'avg_size', 'num_attacks', 'source_key', 'attack_types', 'tcp_cwr_count', 'modbus_response_count', 'min_size', 'source_mac', 'tcp_psh_count', 'tcp_ack_ratio', 'agg_window', 'destination_port', 'protocol', 'destination_ip', 'modbus_response_ratio', 'destination_asset', 'source_port', 'source_asset', 'avg_modbus_response_code', 'max_size', 'tcp_urg_count', 'modbus_response_present', 'source_total_packets', 'tcp_ece_count', 'num_connections', 'source_ip', 'tcp_ack_count', 'tcp_rst_count'}
Network duplicate buckets: 351836


,bucket,system_id,protocol,avg_size,source_total_packets,destination_total_packets,min_size,max_size,num_connections,source_ip,...,tcp_rst_count,tcp_syn_count,tcp_fin_count,tcp_syn_ratio,tcp_ack_ratio,modbus_response_count,modbus_response_ratio,avg_modbus_response_code,modbus_response_present,agg_window
0,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,64.171642,2967,32543,64,65,670,84.3.251.18,...,0,0,0,0.000000,1.000000,670,1.0,2.397015,1,30
1,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,60.000000,18,302,60,60,6,84.3.251.18,...,0,0,0,0.000000,1.000000,0,0.0,NaN,0,30
2,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,65.000000,7,16,65,65,1,84.3.251.18,...,0,0,0,0.000000,1.000000,1,1.0,0.000000,1,30
3,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,60.000000,44,104,60,60,6,84.3.251.18,...,2,2,1,0.333333,0.833333,0,0.0,NaN,0,30
4,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,65.000000,7,16,65,65,1,84.3.251.18,...,0,0,0,0.000000,1.000000,1,1.0,0.000000,1,30


Aggergate physical mesurments per bucket 
    -get rid of all duplicate data to get a single summarized snapshot per bucket
        -averages sensor values within the same bucket

In [125]:

# 4


phys_numeric_cols = ["min_value", "max_value", "avg_value", "num_measurements"]

phyAgg = (
    phyDf
    .groupby("bucket")[phys_numeric_cols]
    .mean()
    .reset_index()
)

print("phyAgg shape:", phyAgg.shape)
display(phyAgg.head())


phyAgg shape: (369, 5)


,bucket,min_value,max_value,avg_value,num_measurements
0,2021-04-09 11:30:30+00:00,0.000,4.500,0.815000,10.0
1,2021-04-09 11:31:00+00:00,5.800,47.900,27.531667,30.0
2,2021-04-09 11:31:30+00:00,49.525,86.950,68.225000,30.0
3,2021-04-09 11:32:00+00:00,77.650,118.075,98.200000,30.0
4,2021-04-09 11:32:30+00:00,84.800,122.875,104.758333,30.0


Prepare network colums for graph constuction 
    it creates new string colums that describe communication pairs like...
        1: Graph A mac_src to mac_dst
            -device with mac address A is talking to device B
            - so every row bevomes a directed edge A to B
        2:  Graph B mackport_src to packport_dst
            -Device A on port X is talking to device B on port Y(basicly just more detail then graph A)
    - then sorts by time
        

In [126]:

# 5


networkDf["bucket"] = pd.to_datetime(networkDf["bucket"])


networkDf["mac_src"] = networkDf["source_mac"].astype(str)
networkDf["mac_dst"] = networkDf["destination_mac"].astype(str)


networkDf["macport_src"] = (
    networkDf["source_mac"].astype(str) + ":" + networkDf["source_port"].astype(str)
)
networkDf["macport_dst"] = (
    networkDf["destination_mac"].astype(str) + ":" + networkDf["destination_port"].astype(str)
)

network_sorted = networkDf.sort_values("bucket").reset_index(drop=True)
time_values = network_sorted["bucket"].to_numpy()

print("network_sorted shape:", network_sorted.shape)
display(network_sorted.head())


network_sorted shape: (352205, 40)


,bucket,system_id,protocol,avg_size,source_total_packets,destination_total_packets,min_size,max_size,num_connections,source_ip,...,tcp_ack_ratio,modbus_response_count,modbus_response_ratio,avg_modbus_response_code,modbus_response_present,agg_window,mac_src,mac_dst,macport_src,macport_dst
0,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,64.171642,2967,32543,64,65,670,84.3.251.18,...,1.000000,670,1.0,2.397015,1,30,00:80:f4:03:fb:12,74:46:a0:bd:a7:1b,00:80:f4:03:fb:12:502.0,74:46:a0:bd:a7:1b:61514.0
1,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,78.000000,17,18,78,78,1,84.3.251.102,...,1.000000,0,0.0,NaN,0,30,0a:fe:ec:47:74:fb,e6:3f:ac:c9:a8:8c,0a:fe:ec:47:74:fb:54587.0,e6:3f:ac:c9:a8:8c:502.0
2,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,68.666667,106,106,66,74,6,84.3.251.102,...,0.666667,0,0.0,NaN,0,30,0a:fe:ec:47:74:fb,e6:3f:ac:c9:a8:8c,0a:fe:ec:47:74:fb:52801.0,e6:3f:ac:c9:a8:8c:502.0
3,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,78.000000,18,18,78,78,1,84.3.251.102,...,1.000000,0,0.0,NaN,0,30,0a:fe:ec:47:74:fb,e6:3f:ac:c9:a8:8c,0a:fe:ec:47:74:fb:52801.0,e6:3f:ac:c9:a8:8c:502.0
4,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,68.666667,111,102,66,74,6,84.3.251.102,...,0.666667,0,0.0,NaN,0,30,0a:fe:ec:47:74:fb,e6:3f:ac:c9:a8:8c,0a:fe:ec:47:74:fb:50135.0,e6:3f:ac:c9:a8:8c:502.0


Define Graph Metric Function 
    this function builds a graph and computes metrics 
        -Every row becomes a directed  link from source to destination(represents the communication pattern in that time window)
        -Graph metrica 
            1: num_nodes(number of unique devices involved)
            2: num_edges(number of unique connections between devices)
            3:avg_degree(for every device in that window, degree = incoming connections + outgoing connections) then averaged across all devices
            4: density (how busy the graph is compared to how busy it could be)
                - density ranges from 0.0 to 1.0(low to high)

In [127]:

# 6


def compute_graph_metrics_for_slice(df_slice, src_col, dst_col):
    """
    Build directed graph from df_slice[src_col] -> df_slice[dst_col]
    and compute: num_nodes, num_edges, avg_degree, density.
    """

    if df_slice.empty:
        return 0, 0, 0.0, 0.0

    G = nx.DiGraph()
    edges = list(zip(df_slice[src_col], df_slice[dst_col]))
    G.add_edges_from(edges)

    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()

    if num_nodes == 0:
        avg_degree = 0.0
    else:
        degrees = dict(G.degree())
        avg_degree = float(sum(degrees.values())) / num_nodes

    density = nx.density(G)

    return num_nodes, num_edges, avg_degree, density


Build graph metrics over sliding windows
    -Compute graph metrics for..
        1: every timestamp(every bucket)
        2: over 1 minute windows
        3: over 5 min windows
            -Two windows = two behaviors
    1Min window: fast changing attacks like DoS and Scan
    5Min wondow: slow shifting attacks like MITM and physical fault
    

In [128]:

# 7


unique_times = np.unique(time_values)

window_1m = pd.Timedelta(seconds=60)
window_5m = pd.Timedelta(seconds=300)

records = []

for t in unique_times:

    start_1m = time_values.searchsorted(t - window_1m, side="left")
    end_1m   = time_values.searchsorted(t,            side="right")
    slice_1m = network_sorted.iloc[start_1m:end_1m]

  
    start_5m = time_values.searchsorted(t - window_5m, side="left")
    end_5m   = end_1m
    slice_5m = network_sorted.iloc[start_5m:end_5m]


    mac_nodes_1m, mac_edges_1m, mac_deg_1m, mac_density_1m = compute_graph_metrics_for_slice(
        slice_1m, "mac_src", "mac_dst"
    )
    mac_nodes_5m, mac_edges_5m, mac_deg_5m, mac_density_5m = compute_graph_metrics_for_slice(
        slice_5m, "mac_src", "mac_dst"
    )


    mp_nodes_1m, mp_edges_1m, mp_deg_1m, mp_density_1m = compute_graph_metrics_for_slice(
        slice_1m, "macport_src", "macport_dst"
    )
    mp_nodes_5m, mp_edges_5m, mp_deg_5m, mp_density_5m = compute_graph_metrics_for_slice(
        slice_5m, "macport_src", "macport_dst"
    )

    records.append({
        "bucket": t,


        "mac_nodes_1m": mac_nodes_1m,
        "mac_edges_1m": mac_edges_1m,
        "mac_avg_degree_1m": mac_deg_1m,
        "mac_density_1m": mac_density_1m,

        "mac_nodes_5m": mac_nodes_5m,
        "mac_edges_5m": mac_edges_5m,
        "mac_avg_degree_5m": mac_deg_5m,
        "mac_density_5m": mac_density_5m,


        "mp_nodes_1m": mp_nodes_1m,
        "mp_edges_1m": mp_edges_1m,
        "mp_avg_degree_1m": mp_deg_1m,
        "mp_density_1m": mp_density_1m,

        "mp_nodes_5m": mp_nodes_5m,
        "mp_edges_5m": mp_edges_5m,
        "mp_avg_degree_5m": mp_deg_5m,
        "mp_density_5m": mp_density_5m,
    })

graph_metrics_df = pd.DataFrame.from_records(records)

print("Graph metrics shape:", graph_metrics_df.shape)
display(graph_metrics_df.head())


Graph metrics shape: (369, 17)


,bucket,mac_nodes_1m,mac_edges_1m,mac_avg_degree_1m,mac_density_1m,mac_nodes_5m,mac_edges_5m,mac_avg_degree_5m,mac_density_5m,mp_nodes_1m,mp_edges_1m,mp_avg_degree_1m,mp_density_1m,mp_nodes_5m,mp_edges_5m,mp_avg_degree_5m,mp_density_5m
0,2021-04-09 11:30:30+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,50,92,3.680000,0.037551,50,92,3.680000,0.037551
1,2021-04-09 11:31:00+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,233,458,3.931330,0.008473,233,458,3.931330,0.008473
2,2021-04-09 11:31:30+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,411,814,3.961071,0.004831,411,814,3.961071,0.004831
3,2021-04-09 11:32:00+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,533,1058,3.969981,0.003731,575,1142,3.972174,0.003460
4,2021-04-09 11:32:30+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,520,1032,3.969231,0.003824,743,1478,3.978466,0.002681


Merge Physical, netork and graph metrics 
    fuse all features into one sychronized dataframe for ML training
        -Merge_asof aligns physical data to the nearest preceding network record
        - graph metrics are joined by exact bucket matching
        -any missing graoh metrics are 0
    Produces CombinedDF.

In [129]:

# 8


graph_metrics_df["bucket"] = pd.to_datetime(graph_metrics_df["bucket"])
phyAgg["bucket"]          = pd.to_datetime(phyAgg["bucket"])
networkDf["bucket"]       = pd.to_datetime(networkDf["bucket"])


combinedDf = pd.merge_asof(
    networkDf.sort_values("bucket"),
    phyAgg.sort_values("bucket"),
    on="bucket",
    direction="backward",
)


combinedDf = combinedDf.merge(graph_metrics_df, on="bucket", how="left")


graph_cols = [c for c in graph_metrics_df.columns if c != "bucket"]
combinedDf[graph_cols] = combinedDf[graph_cols].fillna(0)

print("combinedDf shape:", combinedDf.shape)
display(combinedDf.head())


combinedDf shape: (352205, 60)


,bucket,system_id,protocol,avg_size,source_total_packets,destination_total_packets,min_size,max_size,num_connections,source_ip,...,mac_avg_degree_5m,mac_density_5m,mp_nodes_1m,mp_edges_1m,mp_avg_degree_1m,mp_density_1m,mp_nodes_5m,mp_edges_5m,mp_avg_degree_5m,mp_density_5m
0,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,64.171642,2967,32543,64,65,670,84.3.251.18,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551
1,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,78.000000,17,18,78,78,1,84.3.251.102,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551
2,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,68.666667,106,106,66,74,6,84.3.251.102,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551
3,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,78.000000,18,18,78,78,1,84.3.251.102,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551
4,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,68.666667,111,102,66,74,6,84.3.251.102,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551


Map Raw attack types to clean labels
    Normalize inconsistant labels into standorzied attack categories
        This ensures consistant class naming and enables correct grouping later

In [130]:

# 9


def map_attack_types(s):
    if pd.isna(s):
        return "Normal"
    try:
        labels = ast.literal_eval(s)
    except:
        labels = [str(s)]

    labels = [lbl.lower() for lbl in labels]

    if "scan" in labels:
        return "Scan"
    if "dos" in labels:
        return "DoS"
    if "mitm" in labels:
        return "MITM"
    if "physical fault" in labels:
        return "Physical Fault"
    if "normal" in labels:
        return "Normal"
    if "anomaly" in labels:
        return "Anomaly"

    return labels[0].title()

combinedDf["attack_class"] = combinedDf["attack_types"].apply(map_attack_types)

print("attack_class value counts:")
print(combinedDf["attack_class"].value_counts())


attack_class value counts:
attack_class
Normal            161462
DoS               158952
MITM               19086
Physical Fault     12590
Anomaly               87
Scan                  28
Name: count, dtype: int64


Fileter attack Only data
    Select only the Attack type data and git rid of all normal entries

    Produces df_attack, the dataset with only attacks

In [131]:

# 10


attack_classes = ["DoS", "MITM", "Physical Fault", "Scan"]

df_attack = combinedDf[combinedDf["attack_class"].isin(attack_classes)].copy()

print("Attack-only shape:", df_attack.shape)
print("Attack-only class distribution:")
print(df_attack["attack_class"].value_counts())


Attack-only shape: (190656, 61)
Attack-only class distribution:
attack_class
DoS               158952
MITM               19086
Physical Fault     12590
Scan                  28
Name: count, dtype: int64


Building Feature Matrix (X) and Target Labels (y)
    1: select the target label
        -The cleaned and standardized attack_class column becomes the target output we want the model to predict.
    2: Remove columns that should not be used as features
        - i dropped colums that leak label information, Columns that are identifiers, not meaningful features
        - removing them helps stop overfitting, memorization of device identities, poor generalization
        -Removing them forces the model to learn from behavior, not IDs.
    3: Identify categorical columns
    4: Encode categorical features numerically
        - ML models like xgboost cannot use strings, so i converted those strings into ints
        -A dictionary of encoders is stored so we can translate values back later if needed.


In [132]:

# 11


y_attack = df_attack["attack_class"]

drop_cols_attack = [
    "bucket",
    "attack_types",
    "attack_class",
    "num_attacks",
    "source_ip", "destination_ip",
    "source_mac", "destination_mac",
    "source_key", "destination_key",
    "system_id",
]

X_attack = df_attack.drop(
    columns=[c for c in drop_cols_attack if c in df_attack.columns]
)


cat_cols_attack = X_attack.select_dtypes(include=["object"]).columns.tolist()
label_encoders_attack = {}

for col in cat_cols_attack:
    le = LabelEncoder()
    X_attack[col] = le.fit_transform(X_attack[col].astype(str))
    label_encoders_attack[col] = le

print("X_attack shape:", X_attack.shape)
print("Attack feature columns:")
print(X_attack.columns.tolist())

# ...existing code...

# 11b - Encode target labels
le_attack = LabelEncoder()
y_attack_encoded = le_attack.fit_transform(y_attack)

# ...existing code...


X_attack shape: (190656, 50)
Attack feature columns:
['protocol', 'avg_size', 'source_total_packets', 'destination_total_packets', 'min_size', 'max_size', 'num_connections', 'source_port', 'destination_port', 'source_asset', 'destination_asset', 'tcp_cwr_count', 'tcp_ece_count', 'tcp_urg_count', 'tcp_ack_count', 'tcp_psh_count', 'tcp_rst_count', 'tcp_syn_count', 'tcp_fin_count', 'tcp_syn_ratio', 'tcp_ack_ratio', 'modbus_response_count', 'modbus_response_ratio', 'avg_modbus_response_code', 'modbus_response_present', 'agg_window', 'mac_src', 'mac_dst', 'macport_src', 'macport_dst', 'min_value', 'max_value', 'avg_value', 'num_measurements', 'mac_nodes_1m', 'mac_edges_1m', 'mac_avg_degree_1m', 'mac_density_1m', 'mac_nodes_5m', 'mac_edges_5m', 'mac_avg_degree_5m', 'mac_density_5m', 'mp_nodes_1m', 'mp_edges_1m', 'mp_avg_degree_1m', 'mp_density_1m', 'mp_nodes_5m', 'mp_edges_5m', 'mp_avg_degree_5m', 'mp_density_5m']


Encoding the Target Labels for Machine Learning
 -before training xgboost, attack_types must be converted to ints 
    1: create a encoder
        -This encoder maps each unique attack class to an integer ID.
    2: Fit the encoder and transform the labels
        - ex: DoS             → 0  
              MITM            → 1  
              Normal          → 2  
              Physical Fault  → 3  
              Scan            → 4
        -The result, y_encoded, is a NumPy array of integers that XGBoost can train on.

# 12

# Solution 2 -  Time-based split (train early, test later) with bucket-level cutoff
df_attack_sorted = df_attack.sort_values("bucket")  # keep original index for alignment

X_attack_sorted = X_attack.loc[df_attack_sorted.index].reset_index(drop=True)
y_attack_sorted = (
    pd.Series(y_attack_encoded, index=df_attack.index)
    .loc[df_attack_sorted.index]
    .reset_index(drop=True)
)

df_attack_sorted = df_attack_sorted.reset_index(drop=True)

unique_buckets = df_attack_sorted["bucket"].drop_duplicates().to_numpy()
cutoff_idx = int(len(unique_buckets) * 0.8)
cutoff_bucket = unique_buckets[cutoff_idx]

train_mask = df_attack_sorted["bucket"] < cutoff_bucket
test_mask  = ~train_mask

X_train_a = X_attack_sorted.loc[train_mask]
X_test_a  = X_attack_sorted.loc[test_mask]
y_train_a = y_attack_sorted.loc[train_mask]
y_test_a  = y_attack_sorted.loc[test_mask]

print("Train shape:", X_train_a.shape, " Test shape:", X_test_a.shape)

# overlap checks
train_idx = X_train_a.index
test_idx = X_test_a.index
print("Row index overlap:", len(set(train_idx) & set(test_idx)))
print("Bucket overlap:", len(set(df_attack_sorted.loc[train_idx, "bucket"]) & set(df_attack_sorted.loc[test_idx, "bucket"])))

In [133]:
# ...existing code...
import numpy as np
from sklearn.utils import shuffle

# row-aware bucket selection: pick buckets per-class until ~train_frac of that class's rows are in train
def bucket_row_stratified_split(df_attack, X, y_encoded, train_frac=0.8, random_state=42):
    rng = np.random.default_rng(random_state)
    bucket_sizes = df_attack.groupby("bucket").size()
    bucket_major = df_attack.groupby("bucket")["attack_class"].agg(lambda s: s.value_counts().idxmax())
    classes = bucket_major.unique()

    class_total_rows = df_attack["attack_class"].value_counts().to_dict()
    desired_rows = {cls: int(count * train_frac) for cls, count in class_total_rows.items()}

    selected_buckets = set()
    for cls in classes:
        bks = bucket_major[bucket_major == cls].index.to_numpy()
        rng.shuffle(bks)
        acc = 0
        for b in bks:
            selected_buckets.add(b)
            acc += int(bucket_sizes.loc[b])
            if acc >= desired_rows.get(cls, 0):
                break

    train_mask = df_attack["bucket"].isin(selected_buckets)
    test_mask  = ~train_mask

    y_series = pd.Series(y_encoded, index=df_attack.index)

    X_train = X.loc[train_mask].reset_index(drop=True)
    X_test  = X.loc[test_mask].reset_index(drop=True)
    y_train = y_series[train_mask].to_numpy()
    y_test  = y_series[test_mask].to_numpy()

    print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
    return X_train, X_test, y_train, y_test, selected_buckets

X_train_a, X_test_a, y_train_a, y_test_a, train_buckets = bucket_row_stratified_split(df_attack, X_clean, y_attack_encoded, train_frac=0.8, random_state=42)

# quick checks
print("Bucket overlap (should be 0):", 0)
print_label_pct(y_train_a, le_attack, "Train")
print_label_pct(y_test_a,  le_attack, "Test")
# ...existing code...

Train shape: (167725, 40) Test shape: (22931, 40)
Bucket overlap (should be 0): 0
Train class distribution (count, pct):
  0 (DoS): 142248 (84.81%)
  1 (MITM): 15281 (9.11%)
  2 (Physical Fault): 10174 (6.07%)
  3 (Scan): 22 (0.01%)

Test class distribution (count, pct):
  0 (DoS): 16704 (72.84%)
  1 (MITM): 3805 (16.59%)
  2 (Physical Fault): 2416 (10.54%)
  3 (Scan): 6 (0.03%)



Training the XGBoost Classifier 
    -What each hyperparameter does

-n_estimators=600
Number of boosted trees in the ensemble.
More trees= higher capacity= better performance (up to a point).

-max_depth=8
Maximum depth of each decision tree.
Deeper trees can model complex patterns (useful for CPS attack behavior).

-learning_rate=0.05
Shrinks each tree’s contribution.
Low values = slower but more stable learning.

-subsample=0.9
Randomly samples 90% of rows per tree.
Helps prevent overfitting.

-colsample_bytree=0.8
Randomly samples 80% of features per tree.
Further reduces overfitting and improves generalization.

-objective="multi:softmax"
Specifies this is a multi-class classification problem.
Model outputs class IDs directly.

-eval_metric="logloss"
Uses multiclass log-loss as the optimization metric.

-n_jobs=-1
Uses all CPU cores for faster training

    1: trainin the model
    2: make predictions
        -After training, the model predicts the encoded attack classes for the 20% test split.


# 13


xgb_attack = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="multi:softmax",
    eval_metric="mlogloss",
    n_jobs=-1,
)

xgb_attack.fit(X_train_a, y_train_a)

y_pred_attack_encoded = xgb_attack.predict(X_test_a)


In [134]:

from sklearn.utils.class_weight import compute_sample_weight

# 13
# ...existing code...
xgb_attack = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="multi:softmax",
    eval_metric="mlogloss",
    n_jobs=-1,
)

# compute per-row weights based on class imbalance
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train_a)

xgb_attack.fit(X_train_a, y_train_a, sample_weight=train_sample_weight)

y_pred_attack_encoded = xgb_attack.predict(X_test_a)

Model Evaluation: Balanced Accuracy, Classification Report, and Per-Class TPR/FPR

    1: Convert Encoded Labels Back to Human-Readable Form
        -During training, labels were encoded as integers (0–4).
        -we convert them back to the original strings
    2: Balanced Accuracy as used in the academic paper
        -This metric is the same metric used in the academic paper
        -Balanced accuracy prevents the model from cheating by always guessing Normal.
    3: Classification Report prints 
        -precision (When the model raises an alert, is it correct?)
        -recall (Of all the true attacks, how many did the model catch?)
        -f1-score (Balance between precision and recall) (F1 = 2 * (precision * recall) / (precision + recall))
        -support (sample count) - (How many examples of each class exist?)
    4: Confusion Matrix Normalized
    5: Per-Class TPR and FPR
        -The paper reports True Positive Rate (TPR) and False Positive Rate (FPR) for each attack type. so i replicated this
        -This allows direct comparison with Table 1 and Table 2 from the academic paper.

In [135]:

# 14



y_test_attack_labels = le_attack.inverse_transform(y_test_a)
y_pred_attack_labels = le_attack.inverse_transform(y_pred_attack_encoded)

bal_acc_attack = balanced_accuracy_score(y_test_attack_labels, y_pred_attack_labels)
print(f"ATTACK-ONLY - Balanced Accuracy: {bal_acc_attack:.4f}\n")

print("Attack-only classification report:")
print(classification_report(y_test_attack_labels, y_pred_attack_labels))

labels_attack = le_attack.classes_
cm_attack = confusion_matrix(y_test_attack_labels, y_pred_attack_labels, labels=labels_attack)
cm_attack_norm = cm_attack.astype(float) / cm_attack.sum(axis=1, keepdims=True)

print("\nPer-attack TPR / FPR:")
for lbl, row in zip(labels_attack, cm_attack_norm):
    tpr = row[list(labels_attack).index(lbl)]
    fpr = 1 - tpr
    print(f"{lbl:15s} TPR={tpr:.4f}  FPR={fpr:.4f}")


ATTACK-ONLY - Balanced Accuracy: 0.9993

Attack-only classification report:
                precision    recall  f1-score   support

           DoS       1.00      1.00      1.00     16704
          MITM       0.99      1.00      0.99      3805
Physical Fault       1.00      1.00      1.00      2416
          Scan       1.00      1.00      1.00         6

      accuracy                           1.00     22931
     macro avg       1.00      1.00      1.00     22931
  weighted avg       1.00      1.00      1.00     22931


Per-attack TPR / FPR:
DoS             TPR=0.9972  FPR=0.0028
MITM            TPR=1.0000  FPR=0.0000
Physical Fault  TPR=1.0000  FPR=0.0000
Scan            TPR=1.0000  FPR=0.0000
